In [2]:
import pandas as pd
import numpy as np
from tqdm import tqdm
import insightface
import random
from datetime import datetime

In [4]:
models = ['/home/sho/.insightface/models/buffalo_l/w600k_r50.onnx']
algos = ['brute-force', 'hnsw', 'fvs']

for model in models:
    # get df with embeddings
    for algo in algos:
        if algo == 'brute-force':
            # accpetence_rates = run_brute_force()
            pass
        elif algo == 'hnsw':
            # accpetence_rates = run_hnsw()
            pass
        elif algo == 'fvs':
            # accpetence_rates = run_fvs()
            pass






In [46]:
path_sabrina = "/mnt/nas2/sabrina/face-gen/embeddings_sabrina.pkl"
path = "/mnt/nas2/sabrina/face-gen/embeddings-0312.pkl"
df = pd.read_pickle(path_sabrina)


print(type(df['embedding'].iloc[0]))  # <class 'numpy.ndarray'>
is_array_col = df['embedding'].apply(lambda x: isinstance(x, np.ndarray))
print(is_array_col.value_counts())

# convert back to float32
df['embedding'] = df['embedding'].apply(lambda x: x.astype(np.float32))

<class 'numpy.ndarray'>
embedding
True    13199
Name: count, dtype: int64


In [4]:
df.head()

,name,ID,embedding,embedding type,embedding dtype,embedding_size,image_name,filepath
0,AJ_Cook,0,"[-0.3419863, 0.6712041, -1.2866051, 0.60350394...",<class 'numpy.ndarray'>,float32,512,AJ_Cook_0001.jpg,/home/sho/Insightface-Face-Recognition/img/LFW...
1,AJ_Lamas,1,"[0.059667192, 0.076293096, 1.3340704, 1.245711...",<class 'numpy.ndarray'>,float32,512,AJ_Lamas_0001.jpg,/home/sho/Insightface-Face-Recognition/img/LFW...
2,Aaron_Eckhart,2,"[1.2447274, -0.9747005, 1.2987878, 0.46920043,...",<class 'numpy.ndarray'>,float32,512,Aaron_Eckhart_0001.jpg,/home/sho/Insightface-Face-Recognition/img/LFW...
3,Aaron_Guiel,3,"[-0.17151171, -1.2514898, -0.38039768, -0.0818...",<class 'numpy.ndarray'>,float32,512,Aaron_Guiel_0001.jpg,/home/sho/Insightface-Face-Recognition/img/LFW...
4,Aaron_Patterson,4,"[0.41729146, -0.24729499, -0.86146384, -2.1820...",<class 'numpy.ndarray'>,float32,512,Aaron_Patterson_0001.jpg,/home/sho/Insightface-Face-Recognition/img/LFW...


In [3]:
df['embedding'].iloc[0].dtype

dtype('float32')

In [5]:
detector = insightface.model_zoo.get_model('/home/sho/.insightface/models/buffalo_l/w600k_r50.onnx')


/home/sho/anaconda3/envs/face-gen-notebook/lib/python3.13/site-packages/onnxruntime/capi/onnxruntime_inference_collection.py:118: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(


Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}


In [ ]:
"""query_idx = random.randint(0, len(df) - 1)
query_emb = df['embedding'].iloc[query_idx]
name = df.iloc[query_idx]['name']
person_id = df.iloc[query_idx]['ID']

print(query_idx, name, person_id)

# groupby person id
df[df.ID == person_id].shape[0]"""

11242 Sammy_Sosa 4893


(2, 8)

In [6]:
import platform
platform.node()

'sv7-apu11'

# brute force function

In [56]:
# brute force
def brute_force_search(df):
    # brute force
    top_1_pos = 0
    top_3_pos = 0
    top_10_pos = 0
    total = 0
    top_1_failures = []
    top_3_failures = []
    top_10_failures = []
    search_time = []

    for j in tqdm(range(len(df))):
        """if j >= 100:
            break"""

        query_idx = j
        query_emb = df['embedding'].iloc[query_idx]
        query_person_id = df.iloc[query_idx]['ID']
        query_name = df.iloc[query_idx]['name']

        if df[df.ID == query_person_id].shape[0] == 1:
            continue
        total += 1
        
        # sim scores
        similarities = []

        start_time = datetime.now()
        for i in range(len(df)):
            if i == query_idx:
                continue
            emb = df['embedding'].iloc[i]
            sim = detector.compute_sim(query_emb, emb)
            similarities.append(sim)
        
        #print("top 10:", np.argsort(similarities)[::-1][:10])

        top_10_lst = np.argsort(similarities)[::-1][:10]

        ## === TOP-1 Evaluation ===
        
        top_k_indices = np.argsort(similarities)[::-1][:1]
        top_k_scores = [similarities[i] for i in top_10_lst]

        if df.iloc[top_k_indices[0]]['ID'] == query_person_id:
            top_1_pos += 1

        else:
            top_1_failures.append({
                'query_idx': query_idx,
                'query_id': query_person_id,
                'query_name': query_name,
                'top_k_indices': top_10_lst.tolist(), 
                'top_k_scores':top_k_scores
            })
        end_time = datetime.now()
        search_time.append((end_time - start_time).total_seconds())

        ## === TOP-3 Evaluation ===
        top_k_indices = np.argsort(similarities)[::-1][:3]
        top_k_scores = [similarities[i] for i in top_10_lst]
        top_k_ids = [df.iloc[i]['ID'] for i in top_k_indices]
        if query_person_id in top_k_ids:
            top_3_pos += 1
        else:
            top_3_failures.append({
                'query_idx': query_idx,
                'query_id': query_person_id,
                'query_name': query_name,
                'top_k_indices': top_10_lst.tolist(), 
                'top_k_scores':top_k_scores
            })

        ## === TOP-10 Evaluation ===
        top_k_indices = np.argsort(similarities)[::-1][:10]
        top_k_scores = [similarities[i] for i in top_10_lst]
        top_k_ids = [df.iloc[i]['ID'] for i in top_k_indices]
        if query_person_id in top_k_ids:
            top_10_pos += 1
        else:
            top_10_failures.append({
                'query_idx': query_idx,
                'query_id': query_person_id,
                'query_name': query_name,
                'top_k_indices': top_k_indices.tolist(),
                'top_k_scores:': top_k_scores
            })

    ## Final Reporting
    top_1_acc = top_1_pos/total
    top_3_acc = top_3_pos/total    
    top_10_acc = top_10_pos/total
    latency = np.median(search_time)

    print("Total query attempts:", total)
    print(f"Top-1 Accuracy: {top_1_pos}/{total} = {top_1_pos/total:.4f}")
    print(f"Top-3 Accuracy: {top_3_pos}/{total} = {top_3_pos/total:.4f}")
    print(f"Top-10 Accuracy: {top_10_pos}/{total} = {top_10_pos/total:.4f}")
    print(f"Top-1 Failures: {len(top_1_failures)}")
    print(f"Top-3 Failures: {len(top_3_failures)}")
    print(f"Top-10 Failures: {len(top_10_failures)}")
    print(f"Average Search Time: {latency}")


    return top_k_indices, top_k_scores, top_1_acc, top_3_acc, top_10_acc, latency

In [57]:
top_k_indices, top_k_scores, top_1_acc, top_3_acc, top_10_acc, latency = brute_force_search(df)

100%|██████████| 13199/13199 [33:44<00:00,  6.52it/s]

Total query attempts: 9137
Top-1 Accuracy: 8932/9137 = 0.9776
Top-3 Accuracy: 9003/9137 = 0.9853
Top-10 Accuracy: 9005/9137 = 0.9856
Top-1 Failures: 205
Top-3 Failures: 134
Top-10 Failures: 132
Average Search Time: 0.215531


# hnsw

In [ ]:
"""import hnswlib
import pickle

# === Load embeddings and metadata ===
with open("./embeddings-0312.pkl", "rb") as f:
    df = pickle.load(f)

embedding_matrix = np.stack(df['embedding'].values)
dim = embedding_matrix.shape[1]
print("dim", dim)
print("emb shape", embedding_matrix.shape)

# Build the index
index = hnswlib.Index(space='cosine', dim=dim)
index.init_index(max_elements=len(embedding_matrix), ef_construction=200, M=16)

# Add embeddings to the index
index.add_items(embedding_matrix, np.arange(len(embedding_matrix)))

# Save the index to a binary file
index.save_index("embedding_index_hnsw.bin")

# === Load hnswlib index ===
index = hnswlib.Index(space='cosine', dim=dim)
index.load_index("embedding_index_hnsw.bin")
index.set_ef(50)
"""

dim 512
emb shape (13195, 512)


In [66]:
# hnsw function
import hnswlib

def hnsw_search(df, efc, m, efs):

    embedding_matrix = np.stack(df['embedding'].values)
    dim = embedding_matrix.shape[1]

    # Build the index
    index = hnswlib.Index(space='cosine', dim=dim)
    index.init_index(max_elements=len(embedding_matrix), ef_construction=efc, M=m)

    # set ef search 
    index.set_ef(efs)
    
    # Add embeddings to the index
    index.add_items(embedding_matrix, np.arange(len(embedding_matrix)))

    top_1_pos = 0
    top_3_pos = 0
    top_10_pos = 0
    total = 0

    top_1_failures = []
    top_3_failures = []
    top_10_failures = []
    search_time = []

    # === Evaluation loop ===
    for query_idx in tqdm(range(len(df)), desc="Evaluating search accuracy"):
        """if query_idx >= 100:
            break"""
        
        query_person_id = df.iloc[query_idx]['ID']
        query_name = df.iloc[query_idx]['name']
        
        # Skip if only one image for this person
        if df[df.ID == query_person_id].shape[0] == 1:
            continue
        
        total += 1

        query_embedding = embedding_matrix[query_idx].reshape(1, -1)

        df_copy = df.copy()
        #print("before drop, df copy shape:", df_copy.shape)
        df_copy.drop(query_idx, inplace=True)
        #print("after drop, df copy shape:", df_copy.shape)
        
        embedding_matrix_drop = np.stack(df_copy['embedding'].values)
        
        start_time = datetime.now()
        # Build the index
        index_drop = hnswlib.Index(space='cosine', dim=dim)
        index_drop.init_index(max_elements=len(embedding_matrix_drop), ef_construction=200, M=16)

        # Add embeddings to the index
        index_drop.add_items(embedding_matrix_drop, np.arange(len(embedding_matrix_drop)))
        
        # Search top 10 neighbors (we'll slice top1/top3 from it)
        top_k = 10
        labels, distances = index_drop.knn_query(query_embedding, k=top_k)

        top_k_indices = labels[0]
        top_k_scores = distances[0]
        top_k_ids = [df_copy.iloc[idx]['ID'] for idx in top_k_indices]

        ## Top-1 evaluation
        if top_k_ids[0] == query_person_id:
            top_1_pos += 1
        else:
            top_1_failures.append({
                'query_idx': query_idx,
                'query_id': query_person_id,
                'query_name': query_name,
                'top_k_indices': top_k_indices[:1].tolist(),
                'top_k_ids': top_k_ids[:1],
                'top_k_scores': top_k_scores[:1], 
                'top_k_names': [df_copy.iloc[i]['name'] for i in top_k_indices[:1]]
            })

        end_time = datetime.now()
        search_time.append((end_time - start_time).total_seconds())

        ## Top-3 evaluation
        if query_person_id in top_k_ids[:3]:
            top_3_pos += 1
        else:
            top_3_failures.append({
                'query_idx': query_idx,
                'query_id': query_person_id,
                'query_name': query_name,
                'top_k_indices': top_k_indices[:3].tolist(),
                'top_k_ids': top_k_ids[:3],
                'top_k_scores': top_k_scores[:3], 
                'top_k_names': [df_copy.iloc[i]['name'] for i in top_k_indices[:3]]
            })

        ## Top-10 evaluation
        if query_person_id in top_k_ids[:10]:
            top_10_pos += 1
        else:
            top_10_failures.append({
                'query_idx': query_idx,
                'query_id': query_person_id,
                'query_name': query_name,
                'top_k_indices': top_k_indices.tolist(),
                'top_k_ids': top_k_ids,
                'top_k_scores': top_k_scores, 
                'top_k_names': [df_copy.iloc[i]['name'] for i in top_k_indices]
            })

    # === Final report ===
    top_1_acc = top_1_pos/total
    top_3_acc = top_3_pos/total    
    top_10_acc = top_10_pos/total
    latency = np.median(search_time)
    
    print("Total query attempts:", total)
    print(f"Top-1 accuracy: {top_1_pos}/{total} = {top_1_pos/total:.4f}")
    print(f"Top-3 accuracy: {top_3_pos}/{total} = {top_3_pos/total:.4f}")
    print(f"Top-10 accuracy: {top_10_pos}/{total} = {top_10_pos/total:.4f}")
    print(f"Top-1 failures: {len(top_1_failures)}")
    print(f"Top-3 failures: {len(top_3_failures)}")
    print(f"Top-10 failures: {len(top_10_failures)}")
    print(f"Average Search Time HNSW: {latency}")
    print(f"ef conscontruction = {efc}; M = {m}; ef search = {efs}")

    return top_k_indices, top_k_scores, top_1_acc, top_3_acc, top_10_acc, latency

In [67]:
top_k_indices, top_k_scores, top_1_acc, top_3_acc, top_10_acc, latency = hnsw_search(df, efc=200, m=16, efs=200)

Evaluating search accuracy: 100%|██████████| 13199/13199 [37:49<00:00,  5.82it/s]

Total query attempts: 9137
Top-1 accuracy: 8521/9137 = 0.9326
Top-3 accuracy: 8585/9137 = 0.9396
Top-10 accuracy: 8588/9137 = 0.9399
Top-1 failures: 616
Top-3 failures: 552
Top-10 failures: 549
Average Search Time HNSW: 0.217652
ef conscontruction = 200; M = 16; ef search = 200


# fvs

In [50]:
# fvs search
import fvs_face_helper as fvs

def fvs_search(df):
    top_1_pos, top_3_pos, top_10_pos = 0, 0, 0
    total = 0

    top_1_failures, top_3_failures, top_10_failures = [], [], []
    search_time = []

    #for query_idx in tqdm(range(7350, len(df)), desc="Evaluating search accuracy"):
    for query_idx in tqdm(range(len(df)), desc="Evaluating search accuracy"):
        """if query_idx >= 100:
            break"""
        query_person_id = df.iloc[query_idx]['ID']
        query_name = df.iloc[query_idx]['name']
        query_emb = df['embedding'].iloc[query_idx]
        #print("query idx:", query_idx)
        
        # Skip if only one image for this person
        if df[df.ID == query_person_id].shape[0] == 1:
            continue
        
        total += 1
        #print("query idx:", query_idx)

        similarities = []

        topk = 10
        query_emb = query_emb.reshape((1, 512))
        
        start_time = datetime.now()
        response = fvs.face_search("56d7e7b2-4c17-47a9-b247-3710120b5466", query_emb, topk, verbose=False )
        #print("response=", response)

        # slice idx 1:10 (ignore top 1)
        top_k_indices = response.indices[0][1:] # 9
        top_k_scores = response.distance[0][1:]
        top_k_scores = [float(x) for x in top_k_scores]
    
        #print("top k indicies:", top_k_indices)
        #print("shape:", df.shape)
        top_k_ids = [int(df.iloc[i]['ID']) for i in top_k_indices]
        


        ## Top-1 evaluation
        if df.iloc[top_k_indices[0]]['ID'] == query_person_id:
            #print("top 1 id lst:", df.iloc[top_k_indices[0]]['ID'])
            #print("top 1 person id: ", query_person_id)
            top_1_pos += 1
        else:
            top_1_failures.append({
                'query_idx': query_idx,
                'query_id': query_person_id,
                'query_name': query_name,
                'top_k_indices': top_k_indices[:1], 
                'top_k_scores': top_k_scores[:1], 
                'top_k_names': [df.iloc[i]['name'] for i in top_k_indices[:1]]
            })

        end_time = datetime.now()
        duration = (end_time - start_time).total_seconds()
        search_time.append(duration)

        # 768

        ## Top-3 evaluation
        lst = [df.iloc[i]['ID'] for i in top_k_indices]
        #print("top 3:", lst)
        #print("top 3 person query", query_person_id)
        if query_person_id in lst[:3]:
            top_3_pos += 1
        else:
            top_3_failures.append({
                'query_idx': query_idx,
                'query_id': query_person_id,
                'query_name': query_name,
                'top_k_indices': top_k_indices[:3],
                'top_k_ids': top_k_ids[:3],
                'top_k_scores': top_k_scores[:3], 
                'top_k_names': [df.iloc[i]['name'] for i in top_k_indices[:3]]
            })

        ## Top-10 evaluation
        if query_person_id in lst[:10]:
            top_10_pos += 1
        else:
            top_10_failures.append({
                'query_idx': query_idx,
                'query_id': query_person_id,
                'query_name': query_name,
                'top_k_indices': top_k_indices,
                'top_k_ids': top_k_ids,
                'top_k_scores': top_k_scores, 
                'top_k_names': [df.iloc[i]['name'] for i in top_k_indices]
            })

    # === Final report ===
    top_1_acc = top_1_pos/total
    top_3_acc = top_3_pos/total    
    top_10_acc = top_10_pos/total
    latency = np.median(search_time)

    print("Total query attempts:", total)
    print(f"Top-1 accuracy: {top_1_pos}/{total} = {top_1_pos/total:.4f}")
    print(f"Top-3 accuracy: {top_3_pos}/{total} = {top_3_pos/total:.4f}")
    print(f"Top-10 accuracy: {top_10_pos}/{total} = {top_10_pos/total:.4f}")
    print(f"Top-1 failures: {len(top_1_failures)}")
    print(f"Top-3 failures: {len(top_3_failures)}")
    print(f"Top-10 failures: {len(top_10_failures)}")
    print(f"Average Search Time HNSW: {latency}")

    return top_k_indices, top_k_scores, top_1_acc, top_3_acc, top_10_acc, latency


In [ ]:
top_k_indices, top_k_scores, top_1_acc, top_3_acc, top_10_acc, latency = fvs_search(df)

Evaluating search accuracy: 100%|██████████| 13199/13199 [04:57<00:00, 44.33it/s] 


Total query attempts: 9137
Top-1 accuracy: 8929/9137 = 0.9772
Top-3 accuracy: 9001/9137 = 0.9851
Top-10 accuracy: 9004/9137 = 0.9854
Top-1 failures: 208
Top-3 failures: 136
Top-10 failures: 133
Average Search Time HNSW: 0.03011


| algo | top 1 accuracy | latency (secs) |
| ------ |------ |------ |
| FVS | 97.72% | 0.03011 |
| hnswlib | XX* | XX* |
| brute force | XX | XX |

*could change based on hyperparameters

# compare_faces

In [10]:
from insightface.app import FaceAnalysis

In [11]:
app = FaceAnalysis(name='buffalo_l', providers=['CPUExecutionProvider'])  # Use 'CUDAExecutionProvider' for GPU
app.prepare(ctx_id=-1)  # ctx_id=-1 for CPU, 0 for GPU


Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /home/sho/.insightface/models/buffalo_l/1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /home/sho/.insightface/models/buffalo_l/2d106det.onnx landmark_2d_106 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /home/sho/.insightface/models/buffalo_l/det_10g.onnx detection [1, 3, '?', '?'] 127.5 128.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /home/sho/.insightface/models/buffalo_l/genderage.onnx genderage ['None', 3, 96, 96] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /home/sho/.insightface/models/buffalo_l/w600k_r50.onnx recognition ['None', 3, 112, 112] 127.5 127.5
set det-size: (64

In [12]:
def compare_faces(emb1, emb2, threshold=0.65): # Adjust this threshold according to your usecase.
    """Compare two embeddings using cosine similarity"""
    similarity = np.dot(emb1, emb2) / (np.linalg.norm(emb1) * np.linalg.norm(emb2))
    return similarity, similarity > threshold

In [24]:
emb1 = df['embedding'].iloc[5]
emb2 = df['embedding'].iloc[8]

In [25]:
similarity_score, is_same_person = compare_faces(emb1, emb2)

print(f"Similarity Score: {similarity_score:.4f}")
print(f"Same person? {'YES' if is_same_person else 'NO'}")

Similarity Score: 0.7743
Same person? YES


In [21]:
df.head(10)

,name,ID,embedding,embedding type,embedding dtype,embedding_size,image_name,filepath
0,AJ_Cook,0,"[-0.3419863, 0.6712041, -1.2866051, 0.60350394...",<class 'numpy.ndarray'>,float32,512,AJ_Cook_0001.jpg,/home/sho/Insightface-Face-Recognition/img/LFW...
1,AJ_Lamas,1,"[0.059667192, 0.076293096, 1.3340704, 1.245711...",<class 'numpy.ndarray'>,float32,512,AJ_Lamas_0001.jpg,/home/sho/Insightface-Face-Recognition/img/LFW...
2,Aaron_Eckhart,2,"[1.2447274, -0.9747005, 1.2987878, 0.46920043,...",<class 'numpy.ndarray'>,float32,512,Aaron_Eckhart_0001.jpg,/home/sho/Insightface-Face-Recognition/img/LFW...
3,Aaron_Guiel,3,"[-0.17151171, -1.2514898, -0.38039768, -0.0818...",<class 'numpy.ndarray'>,float32,512,Aaron_Guiel_0001.jpg,/home/sho/Insightface-Face-Recognition/img/LFW...
4,Aaron_Patterson,4,"[0.41729146, -0.24729499, -0.86146384, -2.1820...",<class 'numpy.ndarray'>,float32,512,Aaron_Patterson_0001.jpg,/home/sho/Insightface-Face-Recognition/img/LFW...
5,Aaron_Peirsol,5,"[-0.0011550486, -1.3685023, -0.04704857, 0.508...",<class 'numpy.ndarray'>,float32,512,Aaron_Peirsol_0002.jpg,/home/sho/Insightface-Face-Recognition/img/LFW...
6,Aaron_Peirsol,5,"[-0.3687567, -1.6064845, 1.2511823, -0.3996735...",<class 'numpy.ndarray'>,float32,512,Aaron_Peirsol_0001.jpg,/home/sho/Insightface-Face-Recognition/img/LFW...
7,Aaron_Peirsol,5,"[0.11826757, 0.22257553, 0.58508146, -0.420746...",<class 'numpy.ndarray'>,float32,512,Aaron_Peirsol_0004.jpg,/home/sho/Insightface-Face-Recognition/img/LFW...
8,Aaron_Peirsol,5,"[0.79239154, -0.7837923, 0.58785397, 0.9641951...",<class 'numpy.ndarray'>,float32,512,Aaron_Peirsol_0003.jpg,/home/sho/Insightface-Face-Recognition/img/LFW...
9,Aaron_Pena,6,"[0.36856884, -0.29397953, -0.80316275, -0.6318...",<class 'numpy.ndarray'>,float32,512,Aaron_Pena_0001.jpg,/home/sho/Insightface-Face-Recognition/img/LFW...
